In [1]:
import json
from collections import Counter, defaultdict

def summarize_jsonl_duplicates(jsonl_path):
    normalized_items = []
    line_map = defaultdict(list)

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            norm = json.dumps(obj, ensure_ascii=False, sort_keys=True)
            normalized_items.append(norm)
            line_map[norm].append(line_no)

    counter = Counter(normalized_items)

    duplicates = {k: v for k, v in counter.items() if v > 1}

    print(f"总行数: {len(normalized_items)}")
    print(f"完全重复的 item 数: {sum(v for v in duplicates.values())}")
    print(f"重复组数: {len(duplicates)}")

    for i, (item, cnt) in enumerate(duplicates.items(), 1):
        print(f"\n--- 重复组 {i} ---")
        print(f"出现次数: {cnt}")
        print(f"行号: {line_map[item]}")
        print(f"item: {item[:500]}")  # 太长就截断
        if i >= 10:
            break


summarize_jsonl_duplicates("ego.jsonl")

总行数: 1624
完全重复的 item 数: 0
重复组数: 0


In [9]:
import json
from collections import defaultdict

input_path = "2_hop_without_label_st_se.jsonl"      # 你的原始文件
output_path = "2_hop_without_label_st_se_10.jsonl"  # 过滤后的文件

max_per_answer = 10
answer_count = defaultdict(int)

kept = 0
skipped = 0

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:
    
    for line in fin:
        line = line.strip()
        if not line:
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            print(f"跳过非法 JSON 行: {line[:100]}")
            skipped += 1
            continue

        if "Answer" not in item:
            print(f"跳过缺少 Answer 的 item: {item}")
            skipped += 1
            continue

        answer = item["Answer"]

        if answer_count[answer] < max_per_answer:
            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
            answer_count[answer] += 1
            kept += 1
        else:
            skipped += 1

print("处理完成")
print(f"保留条数: {kept}")
print(f"跳过条数: {skipped}")
print("各 Answer 保留数量:")
for ans, cnt in answer_count.items():
    print(f"{ans}: {cnt}")

处理完成
保留条数: 70
跳过条数: 1554
各 Answer 保留数量:
Neural Networks: 10
Reinforcement Learning: 10
Genetic Algorithms: 10
Probabilistic Methods: 10
Case Based: 10
Theory: 10
Rule Learning: 10
